In [1]:
import os
import subprocess
import pandas as pd

smile_path = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
config_path = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\pitch.conf"

input_folder = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
output_folder = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\features_pitch_new"

os.makedirs(output_folder, exist_ok=True)

wav_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".wav")]

for file in wav_files:
    input_path = os.path.join(input_folder, file)
    output_path = os.path.join(output_folder, file.replace(".wav", ".csv"))

    print(f"\nProcessing: {file}")

    cmd = [
        smile_path,
        "-C", config_path,
        "-I", input_path,
        "-O", output_path
    ]

    print("Command:", cmd)

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    print("Exit code:", result.returncode)
    print("STDERR:", result.stderr)

    # -------------------- FIX DELIMITER IN-PLACE --------------------
    if result.returncode == 0 and os.path.exists(output_path):
        try:
            df = pd.read_csv(output_path, sep=';')  # read using correct delimiter
            df.to_csv(output_path, index=False)     # write back to SAME file
            print("CSV columns fixed (in-place)")
        except Exception as e:
            print("CSV post-processing failed:", e)

print("\nDONE.")



Processing: steth_20181001_11_01_50.wav
Command: ['E:\\Research Project (Prof. Preeti Rao)\\opensmile-3.0-win-x64\\bin\\SMILExtract.exe', '-C', 'E:\\Research Project (Prof. Preeti Rao)\\opensmile-3.0-win-x64\\myconfig\\pitch.conf', '-I', 'E:\\Research Project (Prof. Preeti Rao)\\Top 100 files by Wheeze count\\steth_20181001_11_01_50.wav', '-O', 'E:\\Research Project (Prof. Preeti Rao)\\Classification_24B3907\\features_pitch_new\\steth_20181001_11_01_50.csv']
Exit code: 0
STDERR: (MSG) [2] SMILExtract: openSMILE starting!
(MSG) [2] SMILExtract: config file is: E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\pitch.conf
(MSG) [2] cComponentManager: successfully registered 103 component types.
(WRN) [2] instance 'specScale': FFT magnitude field '*Mag*' not found, defaulting to use full input vector!
(WRN) [2] instance 'specScale': The data type of the input field is not of type DATATYPE_SPECTRUM_BINS_* , spectral power/magnitude bins are required as input to this com

In [17]:
import pandas as pd

features_path = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\features_pitch_new"
txt_to_csv_path = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\txt_to_csv"
output_path = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\Pitch_Shs"

os.makedirs(output_path, exist_ok=True)

# Load feature files normally
features_files = {
    f.replace(".csv", ""): os.path.join(features_path, f)
    for f in os.listdir(features_path)
    if f.endswith(".csv")
}

# Load txt_to_csv files but remove "_label_audacity"
txt_files = {}
for f in os.listdir(txt_to_csv_path):
    if f.endswith(".csv"):
        base_name = f.replace("_label_audacity", "").replace(".csv", "")
        txt_files[base_name] = os.path.join(txt_to_csv_path, f)

# Find matching base names
common = set(features_files.keys()) & set(txt_files.keys())

print(f"Found {len(common)} matching files.")

for name in common:
    try:
        df1 = pd.read_csv(features_files[name])
        df2 = pd.read_csv(txt_files[name])

        # Merge on common column "frame time"
        merged = pd.merge(df1, df2, on="frameTime", how="inner")

        out_file = os.path.join(output_path, name + ".csv")
        merged.to_csv(out_file, index=False)

        print("Merged:", name)

    except Exception as e:
        print("Error in", name, ":", e)

print("\nDone! All merged files saved in 'spec_pitch' folder.")


Found 100 matching files.
Merged: steth_20190817_10_48_27
Merged: steth_20190228_09_54_30
Merged: steth_20190516_14_49_03
Merged: trunc_2019-05-31-15-08-03-L2_14
Merged: steth_20181210_12_31_58
Merged: trunc_2019-07-16-10-41-27-L1_13
Merged: steth_20190720_14_54_03
Merged: steth_20190719_01_26_59
Merged: steth_20181210_09_03_27
Merged: steth_20181210_09_35_29
Merged: steth_20190531_14_15_14
Merged: trunc_2019-05-31-15-38-07-L1_10
Merged: trunc_2019-05-31-14-07-30-L1_13
Merged: steth_20181210_12_30_58
Merged: trunc_2019-05-31-14-07-30-L2_12
Merged: steth_20190228_09_55_19
Merged: trunc_2019-05-31-14-37-49-L2_14
Merged: steth_20190719_01_27_23
Merged: trunc_2019-07-16-10-41-27-L4_13
Merged: trunc_2019-05-31-14-37-49-L6_14
Merged: trunc_2019-05-31-14-37-49-L1_13
Merged: steth_20181001_11_02_11
Merged: steth_20181001_11_01_50
Merged: steth_20190123_08_32_07
Merged: steth_20181108_16_40_02
Merged: steth_20190128_09_29_05
Merged: trunc_2019-05-31-15-38-07-L5_10
Merged: trunc_2019-05-31-14-07

In [20]:
import os
import random
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from sklearn.metrics import precision_recall_curve
from imblearn.over_sampling import SMOTE

# ==============================
# 1. CONFIG
# ==============================
DATA_DIR = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\Pitch_Shs"

FEATURE_COLS = [
    "F0Cand[0]", "F0Cand[1]",
    "candVoicing[0]", "candVoicing[1]",
    "candScores[0]", "candScores[1]"
]

TARGET_COL = "wheeze"

RANDOM_SEED = 42
N_TRAIN_FILES = 80

# ==============================
# 2. LOAD CSV FILE LIST
# ==============================
all_csv_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".csv")
]

if len(all_csv_files) < 100:
    print(f"⚠️ Warning: Found only {len(all_csv_files)} CSV files")

random.seed(RANDOM_SEED)
random.shuffle(all_csv_files)

train_files = all_csv_files[:N_TRAIN_FILES]
test_files = all_csv_files[N_TRAIN_FILES:]

print(f"Training files: {len(train_files)}")
print(f"Testing files : {len(test_files)}")

# ==============================
# 3. HELPER FUNCTION
# ==============================
def load_dataset(file_list):
    X_list, y_list = [], []

    for file in file_list:
        try:
            df = pd.read_csv(file)

            if not all(col in df.columns for col in FEATURE_COLS + [TARGET_COL]):
                print(f"Skipping {os.path.basename(file)} (missing columns)")
                continue

            X_list.append(df[FEATURE_COLS].values)
            y_list.append(df[TARGET_COL].values)

        except Exception as e:
            print(f"Error reading {file}: {e}")

    if not X_list:
        raise RuntimeError("No valid CSV files loaded!")

    return np.vstack(X_list), np.concatenate(y_list)

# ==============================
# 4. LOAD TRAIN & TEST DATA
# ==============================
X_train, y_train = load_dataset(train_files)
X_test, y_test = load_dataset(test_files)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nClass distribution (TRAIN, before SMOTE):")
print(pd.Series(y_train).value_counts())

# ==============================
# 5. APPLY SMOTE (TRAIN ONLY)
# ==============================
smote = SMOTE(
    sampling_strategy="auto",
    random_state=RANDOM_SEED,
    k_neighbors=5
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nClass distribution (TRAIN, after SMOTE):")
print(pd.Series(y_train_sm).value_counts())

# ==============================
# 6. TRAIN XGBOOST
# ==============================
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=RANDOM_SEED
)

model.fit(X_train_sm, y_train_sm)

# ==============================
# 7. THRESHOLD TUNING
# ==============================
y_probs = model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
print(f"\nOptimal decision threshold (F1): {best_threshold:.3f}")
y_pred = (y_probs >= 0.3).astype(int)

# ==============================
# 8. EVALUATION
# ==============================
accuracy = accuracy_score(y_test, y_pred)
precision_v = precision_score(y_test, y_pred, zero_division=0)
recall_v = recall_score(y_test, y_pred, zero_division=0)
f1_v = f1_score(y_test, y_pred, zero_division=0)

print("\n===== TEST METRICS (THRESHOLD TUNED) =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision_v:.4f}")
print(f"Recall   : {recall_v:.4f}")
print(f"F1-score : {f1_v:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred))

# ==============================
# 9. SANITY CHECKS
# ==============================
print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())

print("\nTrue distribution:")
print(pd.Series(y_test).value_counts())

Training files: 80
Testing files : 20

Train shape: (114160, 6)
Test shape : (28540, 6)

Class distribution (TRAIN, before SMOTE):
1    63153
0    51007
Name: count, dtype: int64

Class distribution (TRAIN, after SMOTE):
0    63153
1    63153
Name: count, dtype: int64

Optimal decision threshold (F1): 0.215

===== TEST METRICS (THRESHOLD TUNED) =====
Accuracy : 0.6436
Precision: 0.6483
Recall   : 0.8483
F1-score : 0.7350

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.63      0.36      0.46     11916
           1       0.65      0.85      0.73     16624

    accuracy                           0.64     28540
   macro avg       0.64      0.60      0.60     28540
weighted avg       0.64      0.64      0.62     28540


Prediction distribution:
1    21751
0     6789
Name: count, dtype: int64

True distribution:
1    16624
0    11916
Name: count, dtype: int64
